# Lab 03 — DataFrame API Deep Dive

**Exam domain:** Developing Apache Spark DataFrame/DataSet API Applications — **30% of the exam**, the
single biggest block. Roughly **14 of the 45 questions**.

**Time:** ~3 hours. Split it across two sittings if you need to.

Most exam questions in this domain are *code completion*: a snippet with one or two blanks, four
plausible options. You need the exact method name, the exact argument order, and the exact
`F.something` spelling — with no autocomplete. So: type every cell. Do not copy-paste.

In [ ]:
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("lab03-dfapi").master("local[4]").getOrCreate()

from pyspark.sql import functions as F, Window
from pyspark.sql.types import *

orders = spark.createDataFrame(
    [(1, 101, "NL", "widget,blue,small",  3, 9.99,  1767225600, "2026-01-15 08:31:00"),
     (2, 102, "BE", "widget,red",         1, 9.99,  1767312000, "2026-01-16 12:05:00"),
     (3, 101, "NL", "gadget,green,large", 5, 24.50, 1769904000, "2026-02-02 19:44:00"),
     (4, 103, "DE", "gadget",             2, 24.50, 1770681600, "2026-02-11 07:02:00"),
     (5, 102, "BE", "doodad,blue",        7, 3.25,  1772323200, "2026-03-01 15:20:00"),
     (6, 101, "NL", "widget,blue",        4, 9.99,  1772841600, "2026-03-07 09:00:00"),
     (6, 101, "NL", "widget,blue",        4, 9.99,  1772841600, "2026-03-07 09:00:00"),  # dupe
     (7, 104, None, None,                 1, None,  1772928000, "2026-03-08 11:11:00")],
    "order_id INT, customer_id INT, country STRING, tags STRING, qty INT, "
    "price DOUBLE, epoch LONG, ts STRING")

customers = spark.createDataFrame(
    [(101, "Ada",  "gold"), (102, "Bram", "silver"),
     (103, "Cato", "gold"), (105, "Dries", "bronze")],
    "customer_id INT, name STRING, tier STRING")

orders.show(truncate=False)

## 1. Columns, rows and structure

### Four ways to reference a column (all equivalent)
```python
df.select("qty")            # string
df.select(df.qty)           # attribute
df.select(df["qty"])        # subscript
df.select(F.col("qty"))     # col()  <- the one that always works, incl. after joins
```
Only `F.col()` and `F.expr()` let you build expressions before you have a DataFrame.

### The operations
| Task | Method |
|---|---|
| add / replace a column | `withColumn("new", expr)` |
| add several at once | `withColumns({"a": e1, "b": e2})` (Spark 3.3+) |
| rename | `withColumnRenamed("old", "new")` · `withColumnsRenamed({...})` |
| drop | `drop("a", "b")` — silently ignores names that don't exist |
| keep / project | `select`, `selectExpr("qty * price AS total")` |
| filter rows | `filter(cond)` == `where(cond)`, string or Column |
| limit | `limit(n)` (transformation) vs `take(n)`/`head(n)` (actions) |
| add row number | `F.monotonically_increasing_id()` (unique, **not** consecutive) |

In [ ]:
enriched = (orders
    .withColumn("total",     F.round(F.col("qty") * F.col("price"), 2))
    .withColumn("tag_arr",   F.split(F.col("tags"), ","))
    .withColumn("n_tags",    F.size("tag_arr"))                      # size(null) = -1 !
    .withColumn("first_tag", F.element_at("tag_arr", 1))             # 1-based; getItem(0) is 0-based
    .withColumnRenamed("qty", "quantity")
    .drop("tags"))

enriched.show(truncate=False)
enriched.printSchema()

In [ ]:
# explode family — know the four
arr = enriched.select("order_id", "tag_arr")

print("explode: drops rows whose array is null/empty")
arr.select("order_id", F.explode("tag_arr").alias("tag")).show()

print("explode_outer: keeps them with a null")
arr.select("order_id", F.explode_outer("tag_arr").alias("tag")).show()

print("posexplode: adds the position")
arr.select("order_id", F.posexplode("tag_arr").alias("pos", "tag")).show()
# posexplode_outer = both behaviours

In [ ]:
# Filtering: string SQL vs Column expressions. Both are fine; mixing them is where people slip.
enriched.filter("quantity > 2 AND country = 'NL'").show()
enriched.filter((F.col("quantity") > 2) & (F.col("country") == "NL")).show()   # note: & | ~ and PARENTHESES

# Null handling — `== None` never works. Use isNull / isNotNull / isin / between.
enriched.filter(F.col("country").isNull()).show()
enriched.filter(F.col("country").isin("NL", "BE")).show()
enriched.filter(F.col("quantity").between(2, 5)).show()

### ⚠️ Operator traps that show up on the exam
- `and` / `or` / `not` do **not** work on Columns — use `&`, `|`, `~`, and **wrap each side in parentheses**
  because `&` binds tighter than `==`.
- `F.col("a") == None` is always false; use `.isNull()`.
- `size(null_array)` returns **-1**, not null (unless `spark.sql.legacy.sizeOfNull=false`).
- `element_at` is **1-based**; `getItem`/`[]` on an array is **0-based**.

## 2. Deduplication and validation

In [ ]:
print("rows:", orders.count())
print("distinct() over ALL columns          :", orders.distinct().count())
print("dropDuplicates() == distinct()       :", orders.dropDuplicates().count())
print("dropDuplicates(['customer_id'])      :", orders.dropDuplicates(["customer_id"]).count())
# ^ keeps an ARBITRARY row per key. To control which one, use a window + row_number.

# The deterministic 'latest row per key' pattern — memorise this
w = Window.partitionBy("customer_id").orderBy(F.col("epoch").desc())
latest = (orders.withColumn("rn", F.row_number().over(w))
                .filter(F.col("rn") == 1)
                .drop("rn"))
latest.show()

In [ ]:
# Null handling / validation
print("drop rows with ANY null:");  orders.na.drop().count()
print("drop rows with ALL null:",   orders.na.drop("all").count())
print("drop if <2 non-nulls   :",   orders.na.drop(thresh=2).count())
print("drop if country is null:",   orders.na.drop(subset=["country"]).count())

orders.na.fill({"country": "UNKNOWN", "price": 0.0}).show()
orders.na.replace(["NL"], ["Netherlands"], "country").show()

# coalesce (the FUNCTION — not the partition method!) returns the first non-null
orders.select("order_id", F.coalesce("country", F.lit("??")).alias("country")).show()

> **Name clash worth memorising:** `F.coalesce(col1, col2)` = first non-null value.
> `df.coalesce(n)` = reduce partitions. Same word, unrelated meanings. The exam uses this.

## 3. Aggregations

In [ ]:
agg = (enriched.groupBy("country")
       .agg(F.count("*").alias("n_rows"),                 # counts nulls
            F.count("price").alias("n_prices"),           # SKIPS nulls
            F.countDistinct("customer_id").alias("customers"),
            F.approx_count_distinct("customer_id", 0.05).alias("customers_approx"),
            F.sum("total").alias("revenue"),
            F.avg("quantity").alias("avg_qty"),           # F.mean is an alias of F.avg
            F.min("quantity").alias("min_qty"),
            F.max("quantity").alias("max_qty"),
            F.stddev("quantity").alias("sd_qty"),
            F.collect_list("order_id").alias("orders"),
            F.collect_set("first_tag").alias("tags")))
agg.show(truncate=False)

In [ ]:
# describe vs summary
enriched.describe("quantity", "total").show()      # count, mean, stddev, min, max
enriched.summary().show()                           # + 25%, 50%, 75%  (and you can pass your own)
enriched.summary("count", "min", "50%", "max").show()

# approxQuantile is an ACTION returning a Python list
print(enriched.stat.approxQuantile("total", [0.5, 0.9], 0.01))

In [ ]:
# Multi-level: cube / rollup / pivot
enriched.rollup("country", "first_tag").agg(F.sum("total")).orderBy("country", "first_tag").show()
enriched.cube("country").agg(F.count("*")).show()
enriched.groupBy("country").pivot("first_tag").agg(F.sum("quantity")).show()
# Tip: pivot(col, values_list) is much faster — without the list Spark runs an extra job to find them.

### ⚠️ Aggregation traps
- `count("*")` and `count(1)` count rows including nulls; `count("colname")` **excludes nulls**.
- `countDistinct` is exact and expensive; `approx_count_distinct(col, rsd)` uses HyperLogLog++,
  default `rsd` = 0.05.
- `F.mean` **is** `F.avg`. `F.stddev` is `stddev_samp`; there is also `stddev_pop`.
- After `groupBy(...).agg(...)`, columns not in the grouping or the aggregates are gone.
- `first()`/`last()` in an aggregation are non-deterministic without an ordering, and take an
  `ignorenulls` second argument.

## 4. Dates and timestamps

The exam sub-objective literally says *"epoch conversion and component extraction"*, so drill this.

In [ ]:
d = orders.select("order_id", "epoch", "ts")

d2 = (d
  # string -> timestamp / date
  .withColumn("ts_t",     F.to_timestamp("ts", "yyyy-MM-dd HH:mm:ss"))
  .withColumn("d",        F.to_date("ts", "yyyy-MM-dd HH:mm:ss"))
  # epoch (seconds) -> timestamp, and back
  .withColumn("from_ep",  F.from_unixtime("epoch"))                   # -> STRING
  .withColumn("from_ep_t",F.timestamp_seconds("epoch"))               # -> TIMESTAMP
  .withColumn("to_ep",    F.unix_timestamp("ts", "yyyy-MM-dd HH:mm:ss"))
  # formatting
  .withColumn("fmt",      F.date_format("ts_t", "yyyy/MM/dd")))
d2.show(truncate=False)
d2.printSchema()

In [ ]:
# Component extraction and arithmetic
(d2.select("order_id", "ts_t",
           F.year("ts_t").alias("year"),
           F.quarter("ts_t").alias("q"),
           F.month("ts_t").alias("month"),
           F.dayofmonth("ts_t").alias("dom"),
           F.dayofweek("ts_t").alias("dow"),      # 1 = SUNDAY
           F.dayofyear("ts_t").alias("doy"),
           F.weekofyear("ts_t").alias("woy"),
           F.hour("ts_t").alias("h"), F.minute("ts_t").alias("m"), F.second("ts_t").alias("s"))
   .show())

(d2.select("d",
           F.date_add("d", 7).alias("plus7"),
           F.date_sub("d", 7).alias("minus7"),
           F.add_months("d", 1).alias("next_month"),
           F.last_day("d").alias("month_end"),
           F.next_day("d", "Mon").alias("next_monday"),
           F.trunc("d", "MM").alias("month_start"),          # trunc works on DATE
           F.date_trunc("hour", "ts_t").alias("hour_start"), # date_trunc works on TIMESTAMP
           F.datediff(F.current_date(), "d").alias("days_ago"),
           F.months_between(F.current_date(), "d").alias("months_ago"))
   .show(truncate=False))

### ⚠️ Date traps
- `dayofweek` returns **1 for Sunday**; `weekday` returns 0 for Monday.
- `trunc(date, fmt)` takes the **date first**; `date_trunc(fmt, ts)` takes the **format first**. Reversed.
- `from_unixtime` returns a **STRING**, `timestamp_seconds` returns a **TIMESTAMP**.
- Spark 3 format patterns are strict (`yyyy-MM-dd`, not `YYYY-mm-dd`). `MM` = month, `mm` = minute,
  `DD` = day-of-year, `dd` = day-of-month.
- `to_date`/`to_timestamp` return **null** on unparseable input in the default (`CORRECTED`) mode.

## 5. Joins — expect 2–3 questions

In [ ]:
o = orders.select("order_id", "customer_id", "country", "qty")

for how in ["inner", "left", "right", "outer", "left_semi", "left_anti", "cross"]:
    if how == "cross":
        n = o.crossJoin(customers).count()
    else:
        n = o.join(customers, on="customer_id", how=how).count()
    print(f"{how:10} -> {n} rows")

| `how` | Keeps | Brings right-hand columns? |
|---|---|---|
| `inner` (default) | matches only | ✅ |
| `left` / `left_outer` | all left rows | ✅ (nulls when unmatched) |
| `right` / `right_outer` | all right rows | ✅ |
| `outer` / `full` / `full_outer` | everything | ✅ |
| `left_semi` | left rows **that have** a match | ❌ — left columns only, no duplication |
| `left_anti` | left rows **with no** match | ❌ |
| `cross` | cartesian product | ✅ |

`left_semi` vs `inner`: semi never duplicates a left row even if the right side has several matches, and
returns no right-side columns. That distinction is exam bait.

In [ ]:
# Three join-condition syntaxes
a = o.join(customers, on="customer_id")                                   # string -> ONE customer_id col
b = o.join(customers, o.customer_id == customers.customer_id)             # expr  -> TWO customer_id cols
c = o.join(customers, on=["customer_id"])                                 # list  -> ONE col

print("string:", len(a.columns), "| expr:", len(b.columns), "| list:", len(c.columns))
b.printSchema()      # duplicated column -> later df.select("customer_id") is AMBIGUOUS

In [ ]:
# Multiple keys, and a self-join with aliases (the fix for ambiguity)
left  = orders.alias("l")
right = orders.alias("r")
(left.join(right, (F.col("l.customer_id") == F.col("r.customer_id")) &
                  (F.col("l.order_id")    <  F.col("r.order_id")))
     .select(F.col("l.order_id").alias("first"), F.col("r.order_id").alias("second"))
     .show())

In [ ]:
# unionByName vs union
x = spark.createDataFrame([(1, "a")], "id INT, v STRING")
y = spark.createDataFrame([("b", 2)], "v STRING, id INT")

x.union(y).show()                                  # by POSITION -> garbage, but no error
x.unionByName(y).show()                            # by NAME -> correct
z = spark.createDataFrame([(3,)], "id INT")
x.unionByName(z, allowMissingColumns=True).show()  # fills missing with null

## 6. Broadcast joins

A **broadcast hash join** (a.k.a. map-side join) ships the *small* table to every executor, so the *large*
table never gets shuffled. No `Exchange` on the big side = usually a huge win.

- Automatic when one side's estimated size < `spark.sql.autoBroadcastJoinThreshold`
  (**default 10 MB**; set to `-1` to disable).
- Force it with `F.broadcast(df)` or the SQL hint `/*+ BROADCAST(t) */`.
- The broadcast table is collected **to the driver first**, then sent out — so broadcasting something huge
  OOMs the driver.
- Not available for `full_outer`; the broadcast side must be the one that can be "built"
  (for a `left` join the **right** side is broadcast).
- In the physical plan: `BroadcastHashJoin` + `BroadcastExchange` instead of
  `SortMergeJoin` + two `Exchange`s.

In [ ]:
big   = spark.range(0, 1_000_000).withColumn("customer_id", (F.col("id") % 5 + 101).cast("int"))

print("=== default (small table auto-broadcast) ===")
big.join(customers, "customer_id").explain("formatted")

print("=== broadcast disabled -> SortMergeJoin ===")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
big.join(customers, "customer_id").explain("formatted")

print("=== forced with F.broadcast ===")
big.join(F.broadcast(customers), "customer_id").explain("formatted")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)   # restore

## 7. Window functions

In [ ]:
w_rank    = Window.partitionBy("customer_id").orderBy(F.col("qty").desc())
w_order   = Window.partitionBy("customer_id").orderBy("epoch")          # NO explicit frame
w_running = w_order.rowsBetween(Window.unboundedPreceding, Window.currentRow)

(orders
  .withColumn("rn",         F.row_number().over(w_rank))    # 1,2,3,4  no ties
  .withColumn("rank",       F.rank().over(w_rank))          # 1,2,2,4  gaps
  .withColumn("dense_rank", F.dense_rank().over(w_rank))    # 1,2,2,3  no gaps
  .withColumn("running",    F.sum("qty").over(w_running))   # frame OK for aggregates
  .withColumn("prev_qty",   F.lag("qty", 1).over(w_order))  # lag/lead: window must have NO frame
  .withColumn("next_qty",   F.lead("qty", 1).over(w_order))
  .select("customer_id", "order_id", "qty", "rn", "rank", "dense_rank", "running", "prev_qty", "next_qty")
  .orderBy("customer_id", "order_id")
  .show())

In [ ]:
# ⚠️ Prove the trap: lag/lead over a framed window is an AnalysisException, not a warning.
try:
    orders.withColumn("bad", F.lag("qty").over(w_running)).show()
except Exception as e:
    print(type(e).__name__, "->", str(e).splitlines()[0])

- **`lag`, `lead`, `row_number`, `rank`, `dense_rank`, `ntile` reject an explicit frame** — passing a
  window with `rowsBetween`/`rangeBetween` raises `AnalysisException: Cannot specify window frame for
  lag function`. Keep one framed window for aggregates and one unframed window for ranking/offset
  functions.
- `rowsBetween` counts **physical rows**; `rangeBetween` counts **value offsets** of the ordering column.
- A window with `orderBy` but no explicit frame defaults to
  `rangeBetween(unboundedPreceding, currentRow)` — which is why a "sum over the whole partition" needs an
  explicit `rowsBetween(unboundedPreceding, unboundedFollowing)`.
- Every `Window.partitionBy` is a **shuffle**.

## 8. UDFs, broadcast variables, accumulators

In [ ]:
from pyspark.sql.types import StringType, IntegerType

# --- 1. Plain Python UDF: row-at-a-time, slowest (serialize each row to Python and back) ---
@F.udf(returnType=StringType())
def initials(name):
    return None if name is None else "".join(p[0].upper() for p in name.split())

customers.withColumn("ini", initials("name")).show()

# Alternative registration form, and making it usable from SQL:
upper_udf = F.udf(lambda s: s.upper() if s else None, StringType())
spark.udf.register("upper_udf", upper_udf)              # now callable in spark.sql(...)
customers.createOrReplaceTempView("cust")
spark.sql("SELECT upper_udf(name) FROM cust").show()

In [ ]:
# --- 2. Pandas UDF (vectorized): Arrow-based, batches of rows, MUCH faster ---
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf("double")
def with_vat(price: pd.Series) -> pd.Series:      # Series -> Series
    return price * 1.21

orders.select("order_id", "price", with_vat("price").alias("incl_vat")).show()

@pandas_udf("double")
def total_price(qty: pd.Series, price: pd.Series) -> pd.Series:
    return qty * price
orders.select(total_price("qty", "price").alias("total")).show()

**Why UDFs are the slow path (a guaranteed exam point):** a Python UDF is a **black box** to Catalyst.
No predicate pushdown *through* it, no constant folding, no codegen — and every row is pickled to a Python
worker process and back. Order of preference:

1. **Built-in `pyspark.sql.functions`** — always first choice.
2. **Pandas UDF** — vectorised via Apache Arrow, ~10–100× a plain UDF.
3. **Plain Python UDF** — last resort.

Also: a Python UDF returning the wrong type gives **nulls**, not an error. And UDFs are not
null-safe — Spark calls them with `None` unless you set `asNondeterministic()`/handle it yourself.

In [ ]:
# --- 3. Broadcast variable: a read-only value shipped once per executor ---
sc = spark.sparkContext
rates = sc.broadcast({"NL": 1.0, "BE": 1.05, "DE": 0.98})

@F.udf("double")
def apply_rate(country, price):
    if country is None or price is None:
        return None
    return float(price) * rates.value.get(country, 1.0)

orders.withColumn("local_price", apply_rate("country", "price")).show()

# --- 4. Accumulator: a write-only-by-executors, read-by-driver counter ---
bad_rows = sc.accumulator(0)

@F.udf("int")
def qty_or_count(q):
    if q is None:
        bad_rows.add(1)
        return 0
    return int(q)

orders.select(qty_or_count("qty")).count()
print("accumulator value (driver only):", bad_rows.value)

| | Broadcast variable | Accumulator |
|---|---|---|
| Direction | driver → executors | executors → driver |
| Mutable on executor? | ❌ read-only | ✅ add-only |
| Read on executor? | ✅ `.value` | ❌ (write-only there) |
| Typical use | lookup tables, config, ML models | counters, metrics, bad-record tallies |
| Gotcha | must fit in executor memory; `.unpersist()`/`.destroy()` to free | **only reliable inside actions** — a task retry or a re-executed stage can double-count |

## 9. Actions vs transformations on DataFrames — the exact list

**Actions:** `show`, `count`, `collect`, `take(n)`, `first`, `head(n)`, `tail(n)`, `foreach`,
`foreachPartition`, `toPandas`, `toLocalIterator`, `write.*`, `describe`/`summary` (they run jobs),
`approxQuantile`, `isEmpty`.

**Not actions (lazy):** `select`, `filter`, `withColumn`, `join`, `groupBy`, `orderBy`, `limit`,
`distinct`, `dropDuplicates`, `union`, `repartition`, `coalesce`, `cache`, `persist`, `sample`,
`printSchema` (metadata only — no job), `explain` (no job), `columns`/`schema`/`dtypes` (metadata).

### ✍️ Exercises

**3.1** For each customer, find their most expensive order and the running total of `qty` in order date
sequence. One chain, no subqueries.

**3.2** Return customers who have **never** ordered — two ways: with a `left_anti` join and with a
`left` join + `isNull` filter. Compare the plans.

**3.3** Convert `epoch` to a `TIMESTAMP` column, then produce a monthly revenue table with columns
`year_month` (`yyyy-MM`), `revenue`, `orders`, and `pct_of_total`.

**3.4** Rewrite this as a single expression with no UDF:
```python
@F.udf("string")
def bucketize(q):
    if q is None: return "unknown"
    return "small" if q < 3 else ("medium" if q < 6 else "large")
```

<details><summary>Answers</summary>

**3.2**
```python
customers.join(orders, "customer_id", "left_anti").show()
customers.join(orders, "customer_id", "left").filter(F.col("order_id").isNull()).select(customers["*"]).show()
```
`left_anti` is a single join; the second version materialises all matched rows first, then discards them.

**3.4**
```python
F.when(F.col("qty").isNull(), "unknown") \
 .when(F.col("qty") < 3, "small") \
 .when(F.col("qty") < 6, "medium") \
 .otherwise("large")
```
`when` without a final `otherwise` returns **null** for unmatched rows — remember that.
</details>

## 10. Wrap-up checklist

- [ ] `&`/`|`/`~` with parentheses; never `and`/`or`.
- [ ] `F.coalesce` (first non-null) vs `df.coalesce` (partitions).
- [ ] `count("*")` includes nulls, `count("col")` doesn't.
- [ ] `left_semi` = filter, no right columns, no row duplication.
- [ ] Joining `on="col"` (string/list) gives one column; joining on an expression gives two.
- [ ] `union` is positional, `unionByName` is by name.
- [ ] `broadcast()`, the 10 MB default threshold, and why full outer can't broadcast.
- [ ] `row_number` vs `rank` vs `dense_rank`.
- [ ] `element_at` is 1-based, `getItem` is 0-based, `size(null)` is -1.
- [ ] `trunc(date, fmt)` vs `date_trunc(fmt, ts)` — argument order.
- [ ] Broadcast variable = read-only to executors; accumulator = write-only to executors.
- [ ] Built-in function → pandas UDF → Python UDF, in that order of preference.